# Transactions, Consistency, WAL, and Recovery Demo

This notebook demonstrates:
- Atomic multi-table transactions (all-or-nothing)
- No partial updates after failure
- Consistency checks after each transaction
- WAL records for both transaction and normal (autocommit) operations
- Failure simulation and restart recovery

In [1]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

def find_module_a_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "database").exists():
            return candidate
        alt = candidate / "Assignment-3" / "Module_A"
        if (alt / "database").exists():
            return alt
    raise RuntimeError("Could not locate Assignment-3/Module_A root")

MODULE_A_ROOT = find_module_a_root()
if str(MODULE_A_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_A_ROOT))

from database.db_manager import DatabaseManager
from database.transaction import Transaction

print(f"Module_A root: {MODULE_A_ROOT}")

Module_A root: D:\IITGN\Courses\CS432\project\CS432-Assignments\Assignment-3\Module_A


In [2]:
DEMO_DB_NAME = "txn_demo_notebook_main"
RECOVERY_DB_NAME = "txn_demo_notebook_recovery"

WAL_DIR = MODULE_A_ROOT / "wal"
WAL_DIR.mkdir(parents=True, exist_ok=True)
DEMO_WAL_PATH = WAL_DIR / f"{DEMO_DB_NAME}.wal.jsonl"
RECOVERY_WAL_PATH = WAL_DIR / f"{RECOVERY_DB_NAME}.wal.jsonl"

for wal_path in (DEMO_WAL_PATH, RECOVERY_WAL_PATH):
    if wal_path.exists():
        wal_path.unlink()

def create_schema(db):
    db.create_table(
        name="Members",
        columns=[
            "MemberID",
            "OAUTH_TOKEN",
            "Email",
            "Full_Name",
            "Reputation_Score",
            "Phone_Number",
            "Created_At",
            "Gender",
        ],
        primary_key="MemberID",
        integrity_checks=[
            {"column": "OAUTH_TOKEN", "not_null": True},
            {"column": "Email", "not_null": True, "check": lambda value: value.endswith("@iitgn.ac.in")},
            {"column": "Full_Name", "not_null": True},
            {"column": "Reputation_Score", "not_null": True, "check": lambda value: 0.0 <= value <= 5.0},
            {"column": "Gender", "not_null": True, "check": lambda value: value in {"Male", "Female", "Other"}},
        ],
    )

    db.create_table(
        name="Rides",
        columns=[
            "RideID",
            "Host_MemberID",
            "Start_GeoHash",
            "End_GeoHash",
            "Departure_Time",
            "Vehicle_Type",
            "Max_Capacity",
            "Available_Seats",
            "Base_Fare_Per_KM",
            "Ride_Status",
            "Created_At",
        ],
        primary_key="RideID",
        foreign_keys=[
            {
                "column": "Host_MemberID",
                "references_table": "Members",
                "references_column": "MemberID",
            }
        ],
        integrity_checks=[
            {"column": "Host_MemberID", "not_null": True},
            {"column": "Max_Capacity", "not_null": True, "check": lambda value: value > 0},
            {"column": "Available_Seats", "not_null": True, "check": lambda value: value >= 0},
            {"column": "Available_Seats", "check": lambda value, row: value <= row["Max_Capacity"]},
        ],
    )

    db.create_table(
        name="Bookings",
        columns=[
            "BookingID",
            "RideID",
            "Passenger_MemberID",
            "Booking_Status",
            "Pickup_GeoHash",
            "Drop_GeoHash",
            "Distance_Travelled_KM",
            "Booked_At",
        ],
        primary_key="BookingID",
        foreign_keys=[
            {"column": "RideID", "references_table": "Rides", "references_column": "RideID", "on_delete": "CASCADE"},
            {"column": "Passenger_MemberID", "references_table": "Members", "references_column": "MemberID", "on_delete": "CASCADE"},
        ],
        integrity_checks=[
            {"column": "RideID", "not_null": True},
            {"column": "Passenger_MemberID", "not_null": True},
            {"column": "Distance_Travelled_KM", "not_null": True, "check": lambda value: value > 0},
        ],
    )

def seed_member_1(members, tx=None):
    members.insert_row(
        {
            "MemberID": 1,
            "OAUTH_TOKEN": "tok_alice",
            "Email": "alice@iitgn.ac.in",
            "Full_Name": "Alice",
            "Reputation_Score": 4.7,
            "Phone_Number": "9999999999",
            "Created_At": "2026-01-01 10:00:00",
            "Gender": "Female",
        },
        tx=tx,
    )

def assert_consistency(db):
    members = db.get_table("Members")
    rides = db.get_table("Rides")
    bookings = db.get_table("Bookings")

    member_rows = members.select_all()
    ride_rows = rides.select_all()
    booking_rows = bookings.select_all()

    member_ids = {row["MemberID"] for row in member_rows}
    ride_ids = {row["RideID"] for row in ride_rows}

    for row in ride_rows:
        assert row["Host_MemberID"] in member_ids, f"Broken FK: Rides.Host_MemberID={row['Host_MemberID']}"
        assert row["Available_Seats"] >= 0, "Invalid Available_Seats < 0"
        assert row["Available_Seats"] <= row["Max_Capacity"], "Invalid Available_Seats > Max_Capacity"

    for row in booking_rows:
        assert row["RideID"] in ride_ids, f"Broken FK: Bookings.RideID={row['RideID']}"
        assert row["Passenger_MemberID"] in member_ids, f"Broken FK: Bookings.Passenger_MemberID={row['Passenger_MemberID']}"
        assert row["Distance_Travelled_KM"] > 0, "Invalid distance <= 0"

    return {
        "members": len(member_rows),
        "rides": len(ride_rows),
        "bookings": len(booking_rows),
    }

def wal_summary(wal_path: Path):
    records = Transaction.read_wal_records(str(wal_path))
    type_counts = Counter(record.get("type") for record in records)
    tx_events = defaultdict(list)
    for record in records:
        tx_id = record.get("tx_id")
        if tx_id is not None:
            tx_events[tx_id].append(record.get("type"))
    return records, type_counts, dict(sorted(tx_events.items()))

print("Demo WAL path:", DEMO_WAL_PATH)
print("Recovery WAL path:", RECOVERY_WAL_PATH)

Demo WAL path: D:\IITGN\Courses\CS432\project\CS432-Assignments\Assignment-3\Module_A\wal\txn_demo_notebook_main.wal.jsonl
Recovery WAL path: D:\IITGN\Courses\CS432\project\CS432-Assignments\Assignment-3\Module_A\wal\txn_demo_notebook_recovery.wal.jsonl


## Setup Main Demo Database

Schema creation writes DDL to WAL. The first member insert below is a normal autocommit operation (outside an explicit transaction).

In [3]:
dbm = DatabaseManager()
db = dbm.create_database(DEMO_DB_NAME)
db.set_wal_path(str(DEMO_WAL_PATH))

create_schema(db)
members = db.get_table("Members")
rides = db.get_table("Rides")
bookings = db.get_table("Bookings")

seed_member_1(members)

print("Initial consistency:", assert_consistency(db))
records, type_counts, tx_events = wal_summary(DEMO_WAL_PATH)
print("WAL type counts after setup:", dict(type_counts))
print("Per-tx events so far:")
pprint(tx_events)

Initial consistency: {'members': 1, 'rides': 0, 'bookings': 0}
WAL type counts after setup: {'DDL': 3, 'BEGIN': 1, 'OP': 1, 'COMMIT': 1}
Per-tx events so far:
{1: ['BEGIN', 'OP', 'COMMIT']}


## Atomic Multi-Table Success (All Changes Commit Together)

In [4]:
with db.begin_transaction() as tx:
    rides.insert_row(
        {
            "RideID": 1,
            "Host_MemberID": 1,
            "Start_GeoHash": "dr5ru",
            "End_GeoHash": "dr5rv",
            "Departure_Time": "2026-01-02 11:30:00",
            "Vehicle_Type": "Sedan",
            "Max_Capacity": 4,
            "Available_Seats": 3,
            "Base_Fare_Per_KM": 12.0,
            "Ride_Status": "Open",
            "Created_At": "2026-01-02 11:00:00",
        },
        tx=tx,
    )

    bookings.insert_row(
        {
            "BookingID": 1,
            "RideID": 1,
            "Passenger_MemberID": 1,
            "Booking_Status": "Confirmed",
            "Pickup_GeoHash": "dr5ru",
            "Drop_GeoHash": "dr5rv",
            "Distance_Travelled_KM": 8.5,
            "Booked_At": "2026-01-02 11:15:00",
        },
        tx=tx,
    )

    members.update_row(1, {"Reputation_Score": 4.9}, tx=tx)

assert rides.select(1) is not None
assert bookings.select(1) is not None
assert members.select(1)["Reputation_Score"] == 4.9
print("Committed state:", assert_consistency(db))

Committed state: {'members': 1, 'rides': 1, 'bookings': 1}


## Atomic Multi-Table Failure (No Partial Updates)

A valid member insert is staged first, then a ride insert violates constraints. The full transaction must roll back.

In [5]:
before_rep = members.select(1)["Reputation_Score"]

try:
    with db.begin_transaction() as tx:
        members.insert_row(
            {
                "MemberID": 2,
                "OAUTH_TOKEN": "tok_bob",
                "Email": "bob@iitgn.ac.in",
                "Full_Name": "Bob",
                "Reputation_Score": 4.0,
                "Phone_Number": "8888888888",
                "Created_At": "2026-01-03 10:00:00",
                "Gender": "Male",
            },
            tx=tx,
        )

        rides.insert_row(
            {
                "RideID": 2,
                "Host_MemberID": 2,
                "Start_GeoHash": "dr5ru",
                "End_GeoHash": "dr5rv",
                "Departure_Time": "2026-01-03 11:00:00",
                "Vehicle_Type": "SUV",
                "Max_Capacity": 4,
                "Available_Seats": -1,
                "Base_Fare_Per_KM": 15.0,
                "Ride_Status": "Open",
                "Created_At": "2026-01-03 10:30:00",
            },
            tx=tx,
        )
except Exception as exc:
    print("Expected failure:", type(exc).__name__, str(exc))

assert members.select(2) is None
assert rides.select(2) is None
assert bookings.select(2) is None
assert members.select(1)["Reputation_Score"] == before_rep
print("Rollback verified: no partial updates remain.")
print("Consistent state:", assert_consistency(db))

Expected failure: ValueError CHECK constraint failed: Rides.Available_Seats
Rollback verified: no partial updates remain.
Consistent state: {'members': 1, 'rides': 1, 'bookings': 1}


## WAL for Transactions and Normal Operations

In [6]:
# Normal operations (outside explicit tx) are autocommitted and still logged in WAL.
members.insert_row(
    {
        "MemberID": 99,
        "OAUTH_TOKEN": "tok_temp",
        "Email": "temp@iitgn.ac.in",
        "Full_Name": "Temp User",
        "Reputation_Score": 4.2,
        "Phone_Number": "7000000000",
        "Created_At": "2026-01-05 09:00:00",
        "Gender": "Other",
    }
)
members.update_row(99, {"Reputation_Score": 4.3})
members.delete_row(99)

records, type_counts, tx_events = wal_summary(DEMO_WAL_PATH)
print("WAL type counts:", dict(type_counts))
print("Last 15 WAL record types:", [record.get("type") for record in records[-15:]])
print("Recent transaction event sequences:")
for tx_id in sorted(tx_events.keys())[-6:]:
    print(tx_id, tx_events[tx_id])

WAL type counts: {'DDL': 3, 'BEGIN': 6, 'OP': 8, 'COMMIT': 5, 'ROLLBACK': 1}
Last 15 WAL record types: ['OP', 'OP', 'COMMIT', 'BEGIN', 'OP', 'ROLLBACK', 'BEGIN', 'OP', 'COMMIT', 'BEGIN', 'OP', 'COMMIT', 'BEGIN', 'OP', 'COMMIT']
Recent transaction event sequences:
1 ['BEGIN', 'OP', 'COMMIT']
2 ['BEGIN', 'OP', 'OP', 'OP', 'COMMIT']
3 ['BEGIN', 'OP', 'ROLLBACK']
4 ['BEGIN', 'OP', 'COMMIT']
5 ['BEGIN', 'OP', 'COMMIT']
6 ['BEGIN', 'OP', 'COMMIT']


## Failure Handling and Recovery After Restart

We simulate a crash by leaving one transaction incomplete (no commit/rollback), then recovering from WAL in a new database manager instance.

In [7]:
live_manager = DatabaseManager()
live_db = live_manager.create_database(RECOVERY_DB_NAME)
live_db.set_wal_path(str(RECOVERY_WAL_PATH))
create_schema(live_db)

live_members = live_db.get_table("Members")
live_rides = live_db.get_table("Rides")
live_bookings = live_db.get_table("Bookings")
seed_member_1(live_members)

# Committed transaction (must survive restart).
with live_db.begin_transaction() as tx:
    live_rides.insert_row(
        {
            "RideID": 10,
            "Host_MemberID": 1,
            "Start_GeoHash": "dr5sa",
            "End_GeoHash": "dr5sb",
            "Departure_Time": "2026-02-01 08:30:00",
            "Vehicle_Type": "Sedan",
            "Max_Capacity": 4,
            "Available_Seats": 2,
            "Base_Fare_Per_KM": 10.0,
            "Ride_Status": "Open",
            "Created_At": "2026-02-01 08:00:00",
        },
        tx=tx,
    )

    live_bookings.insert_row(
        {
            "BookingID": 10,
            "RideID": 10,
            "Passenger_MemberID": 1,
            "Booking_Status": "Confirmed",
            "Pickup_GeoHash": "dr5sa",
            "Drop_GeoHash": "dr5sb",
            "Distance_Travelled_KM": 5.0,
            "Booked_At": "2026-02-01 08:10:00",
        },
        tx=tx,
    )

# Incomplete transaction (simulated crash before commit).
tx_incomplete = live_db.begin_transaction()
live_members.insert_row(
    {
        "MemberID": 20,
        "OAUTH_TOKEN": "tok_incomplete",
        "Email": "incomplete@iitgn.ac.in",
        "Full_Name": "Incomplete User",
        "Reputation_Score": 3.8,
        "Phone_Number": "7222222222",
        "Created_At": "2026-02-02 09:00:00",
        "Gender": "Other",
    },
    tx=tx_incomplete,
)
live_rides.insert_row(
    {
        "RideID": 20,
        "Host_MemberID": 20,
        "Start_GeoHash": "dr5sc",
        "End_GeoHash": "dr5sd",
        "Departure_Time": "2026-02-02 09:30:00",
        "Vehicle_Type": "SUV",
        "Max_Capacity": 4,
        "Available_Seats": 3,
        "Base_Fare_Per_KM": 11.0,
        "Ride_Status": "Open",
        "Created_At": "2026-02-02 09:10:00",
    },
    tx=tx_incomplete,
)
# tx_incomplete.commit()

print("Before simulated crash:")
print("- committed ride 10 visible:", live_rides.select(10) is not None)
print("- incomplete member 20 visible outside tx:", live_members.select(20) is not None)
print("- incomplete member 20 visible inside tx:", live_members.select(20, tx=tx_incomplete) is not None)

# Simulate process death by abandoning in-memory state without commit/rollback.
del tx_incomplete
del live_db
del live_manager

Before simulated crash:
- committed ride 10 visible: True
- incomplete member 20 visible outside tx: False
- incomplete member 20 visible inside tx: True


In [8]:
restart_manager = DatabaseManager()
recovered_db = restart_manager.create_database(RECOVERY_DB_NAME)
recovered_db.set_wal_path(str(RECOVERY_WAL_PATH))
applied_ops = recovered_db.recover_from_wal()

rec_members = recovered_db.get_table("Members")
rec_rides = recovered_db.get_table("Rides")
rec_bookings = recovered_db.get_table("Bookings")

assert rec_rides.select(10) is not None
assert rec_bookings.select(10) is not None

assert rec_members.select(20) is None
assert rec_rides.select(20) is None
assert rec_bookings.select(20) is None

print("Applied committed OP records during recovery:", applied_ops)
print("Recovered consistent state:", assert_consistency(recovered_db))
print("Recovery result: committed data retained, incomplete transaction discarded.")

records, _, _ = wal_summary(RECOVERY_WAL_PATH)
begins = {record.get("tx_id") for record in records if record.get("type") == "BEGIN"}
commits = {record.get("tx_id") for record in records if record.get("type") == "COMMIT"}
rollbacks = {record.get("tx_id") for record in records if record.get("type") == "ROLLBACK"}
incomplete_tx_ids = sorted(begins - commits - rollbacks)
print("Incomplete tx IDs present in WAL:", incomplete_tx_ids)

Applied committed OP records during recovery: 3
Recovered consistent state: {'members': 1, 'rides': 1, 'bookings': 1}
Recovery result: committed data retained, incomplete transaction discarded.
Incomplete tx IDs present in WAL: [3]
